# Data Validaton

In [ ]:
import pandas as pd

mes = pd.read_csv("../data/mes_events.csv")
erp = pd.read_csv("../data/erp_work_orders.csv")
specs = pd.read_csv("../data/eng_specs.csv")



In [45]:
mes.event_time = pd.to_datetime(mes.event_time)
mes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   event_id        3 non-null      int64         
 1   serial_number   3 non-null      object        
 2   process_step    3 non-null      object        
 3   measured_value  3 non-null      float64       
 4   event_time      3 non-null      datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 252.0+ bytes


In [47]:
mes

,event_id,serial_number,process_step,measured_value,event_time
0,1,A1001,Autoclave_Cure,182.50,2026-01-01 10:05:00
1,2,A1001,Autoclave_Cure,195.20,2026-01-01 10:15:00
2,3,A1002,CNC_Milling,0.42,2026-01-01 11:20:00


In [38]:
erp

,work_order_id,serial_number,part_number
0,WO-001,A1001,P-9001
1,WO-002,A1002,P-9002


In [39]:
specs

,part_number,process_step,lower_limit,upper_limit
0,P-9001,Autoclave_Cure,175.00,190.00
1,P-9002,CNC_Milling,0.35,0.45


#### Join MES + ERP + Specs

In [40]:
df = mes.merge(erp, how="left", on="serial_number").merge(specs, how="left", on=["part_number","process_step"])
df

,event_id,serial_number,process_step,measured_value,event_time,work_order_id,part_number,lower_limit,upper_limit
0,1,A1001,Autoclave_Cure,182.50,2026-01-01 10:05:00,WO-001,P-9001,175.00,190.00
1,2,A1001,Autoclave_Cure,195.20,2026-01-01 10:15:00,WO-001,P-9001,175.00,190.00
2,3,A1002,CNC_Milling,0.42,2026-01-01 11:20:00,WO-002,P-9002,0.35,0.45


## Validation logic

In [ ]:
df["out_of_spec"] = (
    (df["measured_value"] < df["lower_limit"]) |
    (df["measured_value"] > df["upper_limit"])
)
df_flag = df[df["out_of_spec"] == True][[
    "serial_number",
    "process_step",
    "measured_value",
    "lower_limit",
    "upper_limit",
    "out_of_spec"
]]

In [42]:
df_flag

,serial_number,process_step,measured_value,lower_limit,upper_limit,out_of_spec
1,A1001,Autoclave_Cure,195.2,175.0,190.0,True


## Consecutive failures

In [28]:
df = df.sort_values(["serial_number", "event_time"])
df["consecutive_failures"] = (
    df.groupby("serial_number")["out_of_spec"]
      .transform(lambda x: x.astype(int).groupby(x.eq(False).cumsum()).cumsum())
)

df

,event_id,serial_number,process_step,measured_value,event_time,work_order_id,part_number,lower_limit,upper_limit,out_of_spec,consecutive_failures
0,1,A1001,Autoclave_Cure,182.50,2026-01-01 10:05:00,WO-001,P-9001,175.00,190.00,False,0
1,2,A1001,Autoclave_Cure,195.20,2026-01-01 10:15:00,WO-001,P-9001,175.00,190.00,True,1
2,3,A1002,CNC_Milling,0.42,2026-01-01 11:20:00,WO-002,P-9002,0.35,0.45,False,0
